## Load the readings and metadata


In [1]:
import pandas as pd

cpu = pd.read_csv("../challenge_data/vm_cpu_readings.csv.gz")
meta = pd.read_csv("../challenge_data/vm_metadata.csv.gz")

print(cpu.shape, meta.shape)
print(cpu.dtypes)
print(cpu.head())
print(cpu.isna().sum())
print(cpu.describe())

(1340000, 5) (10000, 11)
timestamp      int64
vm_id            str
min_cpu      float64
max_cpu      float64
avg_cpu      float64
dtype: object
   timestamp                                              vm_id   min_cpu  \
0          0  H+FUHUNaeQQvSYHWhak3XW+9PifF0TXQiyYQGEVbDiJoZP...  1.812194   
1          0  pjJUurJZ7RpWz3MS+oYTDl0CBOeltBhkx7g/e1oB5V6wiW...  4.126021   
2          0  iiljldVGhxG8yYxCrv3jGPpNnvhepXkq21OWoaI9RwsN+Q...  1.225669   
3          0  /f+jE9mriEjSsdEzv/qBP0jhMk+EYf9qw928F/sgbwOX2F...  1.939634   
4          0  wClSIZ+AWnTzhVNn/JqFn6WVHZDJ4lJ3aaL2NC8ZqNfUlu...  0.352236   

     max_cpu   avg_cpu  
0  14.587021  4.851659  
1   5.650850  4.687886  
2   2.082111  1.607588  
3   3.228830  2.374714  
4   0.855003  0.597535  
timestamp    0
vm_id        0
min_cpu      0
max_cpu      0
avg_cpu      0
dtype: int64
          timestamp       min_cpu       max_cpu       avg_cpu
count  1.340000e+06  1.340000e+06  1.340000e+06  1.340000e+06
mean   1.995000e+04  6.156478e+

**What this shows:** The files loaded successfully: there are 1.34 million CPU readings for 10,000 VMs, with five reading fields and eleven metadata fields. No CPU reading values are missing. The summary also shows a wide range of VM usage, so later checks can focus on whether the records are internally consistent.


## Check timestamp coverage and duplicate records


In [2]:
print(cpu["timestamp"].nunique())
print(sorted(cpu["timestamp"].unique())[:5])
print(cpu["timestamp"].drop_duplicates().sort_values().diff().value_counts())
print(cpu.duplicated(["vm_id", "timestamp"]).sum())
print(cpu.groupby("vm_id").size().value_counts())

134
[np.int64(0), np.int64(300), np.int64(600), np.int64(900), np.int64(1200)]
timestamp
300.0    133
Name: count, dtype: int64
0
134    10000
Name: count, dtype: int64


**What this shows:** The readings cover 134 distinct timestamps (0 to 39,900 s), and every gap between consecutive timestamps is exactly 300 s (five minutes), so no timestamps are missing. Every VM has exactly 134 readings, and there are no duplicate VM-and-timestamp pairs. The table is complete and evenly spaced, so no resampling or gap filling is needed.

## Validate CPU values and their ordering


In [3]:
import numpy as np

vals = cpu[["min_cpu", "avg_cpu", "max_cpu"]]
print("non-finite:", (~np.isfinite(vals)).sum().to_dict())
print("outside [0, 100]:", ((vals < 0) | (vals > 100)).sum().to_dict())

# min <= avg <= max should hold in every row
bad = (cpu["min_cpu"] > cpu["avg_cpu"]) | (cpu["avg_cpu"] > cpu["max_cpu"])
print("rows where min/avg/max order is broken:", int(bad.sum()))

non-finite: {'min_cpu': 0, 'avg_cpu': 0, 'max_cpu': 0}
outside [0, 100]: {'min_cpu': 0, 'avg_cpu': 0, 'max_cpu': 0}
rows where min/avg/max order is broken: 0


**What this shows:** All minimum, average, and maximum CPU values are finite and fall between 0% and 100%. Every row also follows the expected order: minimum CPU is no greater than average CPU, which is no greater than maximum CPU. Every row is therefore a valid observation and no rows need to be removed.


## Check that metadata and readings describe the same VMs

In [4]:
print("metadata vm_id unique:", meta["vm_id"].is_unique)
print("same VM set in both files:", set(meta["vm_id"]) == set(cpu["vm_id"]))

metadata vm_id unique: True
same VM set in both files: True


**What this shows:** Each VM ID appears exactly once in the metadata, and the metadata and the readings contain the same set of 10,000 VMs. The segment join in notebook 02 therefore cannot drop or duplicate any VM.

## Data card claims versus these checks

| Data card statement | Checked here | Result |
| --- | --- | --- |
| 10,000 VMs, 134 measurements each, 1,340,000 rows | Shape and rows per VM | Confirmed |
| Consecutive measurements are exactly 300 s apart | Gaps between sorted timestamps | Confirmed |
| No duplicate `(vm_id, timestamp)` pairs | Duplicate count | Confirmed (0) |
| One metadata row per VM and matching VM populations | Unique IDs and set comparison | Confirmed |
| Category counts 6,180 / 3,091 / 729 | Counted in notebook 02 | Confirmed |

The data card does not define the units or special values of `timestamp_vm_created` and `timestamp_vm_deleted`, so notebook 02 treats their interpretation as an assumption.

## Audit summary

- The data is complete and clean: 1,340,000 readings, 10,000 VMs × 134 timestamps, regular five-minute spacing, no missing timestamps and no duplicate VM-and-timestamp pairs.
- There are no missing, non-finite or out-of-range CPU values, and minimum ≤ average ≤ maximum holds in every row.
- The metadata and the readings describe the same 10,000 VMs.
- **Preprocessing decision:** no cleaning, imputation or resampling is needed. The target builder still keeps only rows with a finite `avg_cpu` in [0, 100] and one row per VM and timestamp, so it stays correct if it is run on other data.
- **Caveat from the data card:** only VMs with complete traces over the whole window were eligible, and the 10,000 VMs were chosen with stratification by VM category and virtual-core bucket. So the data cannot show VM arrivals or departures, and segment shares reflect the selection, not the full Azure population.
- Next: notebook 02 builds the two targets and analyses the training window only.